# initialiastion du projet de detection de la pneumonie 

In [ ]:
import tensorflow as tf # Bibliothèque principale pour la création et l'entraînement de modèles de Deep Learning
from tensorflow.keras import layers, models # Importation des outils pour construire les couches (Conv2D, Dense) et la structure (Sequential)
from tensorflow.keras.utils import plot_model # Outil pour générer un schéma visuel de l'architecture de votre réseau de neurones
import matplotlib.pyplot as plt # Bibliothèque pour tracer les graphiques (Accuracy, Loss) et afficher les images médicales
import numpy as np # Outil de calcul numérique pour manipuler les données (tableaux d'images et de labels)
from sklearn.metrics import classification_report # Outil pour calculer la Précision, le Rappel (Recall) et le F1-Score
from sklearn.metrics import confusion_matrix # Importation spécifique pour créer la matrice de confusion
import seaborn as sns # Bibliothèque de visualisation pour rendre la matrice de confusion plus lisible et esthétiques
import os # Bibliothèque pour interagir avec le système d'exploitation (gestion des dossiers et des fichiers d'images)


## Fonction pour afficher un certain nombre d’images d’un dossier

In [ ]:
# Import des bibliothèques nécessaires
import matplotlib.pyplot as plt
import os
from PIL import Image

# Fonction pour afficher un certain nombre d’images d’un dossier
def afficher_images(directory, class_name, n=4):
    print(f"\n--- {n} images pour {class_name} class ---")
    images_to_show = []
    # Parcours des fichiers dans le dossier
    for filename in os.listdir(directory):

        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            images_to_show.append(os.path.join(directory, filename))
        if len(images_to_show) == n:
            break

    plt.figure(figsize=(10, 5))
    for i, img_path in enumerate(images_to_show):
        img = Image.open(img_path)
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"{class_name} {i+1}")
        plt.axis('off')

    # Affichage final
    plt.show()


## Analyse des classes

In [ ]:
# Analyse des classes (Comptage du nombre d’images par classe)
Nbr_Nor = len(os.listdir(chemin+"/chest_xray/train/NORMAL"))
Nbr_Pneu= len(os.listdir(chemin+"/chest_xray/train/PNEUMONIA"))
print("Images NORMAL :", Nbr_Nor)
print("Images PNEUMONIA :", Nbr_Pneu)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


classes = ['NORMAL', 'PNEUMONIA']
counts = [Nbr_Nor, Nbr_Pneu]

plt.figure(figsize=(8, 6))
plt.bar(classes, counts, color=['skyblue', 'lightcoral'])
plt.xlabel('Classes')
plt.ylabel("Nombre d'images")
plt.title("Distribution des images par classe avant augmentation")
plt.show()

In [ ]:
# taille des images de la classe NORMAL soucre
from PIL import Image #

normal_dir = chemin + "/chest_xray/train/NORMAL"

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

In [ ]:
# taille des images de la classe NORMAL soucre
from PIL import Image #

normal_dir = chemin + "/chest_xray/train/NORMAL"

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

# Generation methode de Rotation(ImageDataGenerator de tensorflow)

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

normal_dir = chemin + "/chest_xray/train/NORMAL"
extensions = ('.jpg', '.jpeg', '.png')

# Nombre actuel d’images
current_count = len([f for f in os.listdir(normal_dir) if f.lower().endswith(extensions)])

target = 3875 # Objectif de 3875 images pour la classe NORMAL (équilibrage avec la classe PNEUMONIA)
generated = 0

original_images = [f for f in os.listdir(normal_dir) if f.lower().endswith(extensions)]

# Définir un répertoire accessible en écriture pour les images augmentées
save_augmented_dir = '/content/normal_rotation_augmente'
os.makedirs(save_augmented_dir, exist_ok=True)

for img_name in original_images:

    if current_count + generated >= target:
        break

    img_path = os.path.join(normal_dir, img_name)
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = x.reshape((1,) + x.shape)

    for batch in datagen.flow(
            x,
            batch_size=1,
            save_to_dir=save_augmented_dir, # Modifié en répertoire accessible en écriture
            save_prefix='aug',
            save_format='jpeg'):

        generated += 1

        if current_count + generated >= target:
            break

print("Total NORMAL :", current_count + generated)

## nombres d'image generer

In [ ]:
# Verication du nombre d'image generer
Nbr_Nor = len(os.listdir("/content/normal_rotation_augmente"))
print("Images NORMAL :", Nbr_Nor)

## taille des images de la classe NORMAL augmentee


In [ ]:
# taille des images de la classe NORMAL augmentee
from PIL import Image #

normal_dir = "/content/normal_rotation_augmente" # chemin en chemin vers les images augmentee

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## Affichage des images de la classe Normal du Data Augmentation

In [ ]:
# Affichage de 5 images de la classe NORMAL augmentee
train_dir = "/content/normal_rotation_augmente"
# affichage des images de la classe NORMAL
afficher_images(train_dir, "NORMAL ROT",5)

### Étape 1 du GAN : Chargement des images et prétraitement

In [ ]:
# Étape 1 : Chargement des images et prétraitement
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

IMG_SIZE = 224  # taille des images
LATENT_DIM = 100  # dimension du vecteur latent
BATCH_SIZE = 32  # taille du batch
EPOCHS = 1000  # nombre d’époques

normal_dir = chemin + "/chest_xray/train/NORMAL"  # dossier NORMAL
target = 3875  # objectif d’images
images = []  # stockage des images

# parcourir les fichiers du dossier
for file in os.listdir(normal_dir):

    # vérifier que le fichier est une image
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = load_img(os.path.join(normal_dir, file), target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale")  # chargement + resize + grayscale
        img = img_to_array(img)  # conversion numpy
        images.append(img)  # ajout à la liste

images = np.array(images, dtype=np.float32)  # conversion en array

# vérifier qu’il y a des images
if len(images) == 0:
    raise ValueError(f"Aucune image trouvée dans {normal_dir}")

images = (images - 127.5) / 127.5  # normalisation [-1,1]

### Étape 2 du GAN — Générateur (sortie : 224x224x1)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 2 — Générateur (sortie : 224x224x1)
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),  # vecteur latent en entrée

        layers.Dense(7 * 7 * 256, use_bias=False),  # projection dense initiale
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation

        layers.Reshape((7, 7, 256)),  # reshape en feature map

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # 14x14
        layers.BatchNormalization(),  # normalisation
        layers.LeakyReLU(),  # activation

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),  # 28x28
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(32, 5, strides=2, padding="same", use_bias=False),  # 56x56
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(16, 5, strides=2, padding="same", use_bias=False),  # 112x112
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, 5, strides=2, padding="same", use_bias=False, activation="tanh")  # 224x224x1
    ])
    return model

### Étape 3 du GAN — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)

In [ ]:
# Étape 3 — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),  # image en entrée

        layers.Conv2D(32, 5, strides=2, padding="same"),  # réduction taille
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(128, 5, strides=2, padding="same"),  # extraction features
        layers.BatchNormalization(),  # stabilisation (optionnel)
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, 5, strides=2, padding="same"),  # features profondes
        layers.BatchNormalization(),  # stabilisation (optionnel)
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),  # mise à plat
        layers.Dense(1)  # sortie (réel/faux)
    ])
    return model